# 📡 Monitoreo de Modelo y Detección de Data Drift

**Objetivo:** Implementar sistema de monitoreo para detectar degradación del modelo en producción.

Este notebook incluye:
- Detección de Data Drift usando múltiples métricas
- Kolmogorov-Smirnov Test (KS Test)
- Population Stability Index (PSI)
- Jensen-Shannon Divergence
- Análisis de distribuciones de características
- Reportes y recomendaciones

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import joblib
from datetime import datetime

# Estadística
from scipy import stats
from scipy.spatial.distance import jensenshannon
from scipy.stats import ks_2samp, chi2_contingency

warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 📥 1. Cargar Datos de Referencia y Producción

In [ ]:
# Cargar dataset completo (simularemos datos de referencia vs producción)
ruta_datos = os.path.join('..', '..', 'base_de_datos.csv')
df_completo = pd.read_csv(ruta_datos)

print(f"✅ Dataset cargado: {df_completo.shape}")

# Detectar variable objetivo
posibles_targets = [col for col in df_completo.columns if any(keyword in col.lower() for keyword in 
                    ['target', 'label', 'exited', 'churn', 'class'])]
target_col = posibles_targets[0] if posibles_targets else None

if target_col:
    print(f"Variable objetivo: {target_col}")
else:
    print("⚠️ No se detectó variable objetivo")

In [ ]:
# Simular escenario: Dividir en datos de referencia (80%) y datos de producción (20%)
from sklearn.model_selection import train_test_split

df_referencia, df_produccion = train_test_split(
    df_completo, 
    test_size=0.2, 
    random_state=42,
    stratify=df_completo[target_col] if target_col else None
)

print(f"\n📊 División de datos:")
print(f"  - Referencia (baseline): {df_referencia.shape[0]} filas")
print(f"  - Producción (nuevo): {df_produccion.shape[0]} filas")

## 🔍 2. Funciones de Detección de Drift

In [ ]:
def kolmogorov_smirnov_test(ref_data, prod_data, threshold=0.05):
    """Prueba de Kolmogorov-Smirnov para detectar drift"""
    ref_clean = ref_data.dropna()
    prod_clean = prod_data.dropna()
    
    statistic, p_value = ks_2samp(ref_clean, prod_clean)
    
    return {
        'statistic': statistic,
        'p_value': p_value,
        'drift_detected': p_value < threshold,
        'threshold': threshold
    }


def population_stability_index(ref_data, prod_data, n_bins=10):
    """Calcula el Population Stability Index (PSI)"""
    ref_clean = ref_data.dropna()
    prod_clean = prod_data.dropna()
    
    breakpoints = np.linspace(ref_clean.min(), ref_clean.max(), n_bins + 1)
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf
    
    ref_counts = pd.cut(ref_clean, bins=breakpoints).value_counts().sort_index()
    prod_counts = pd.cut(prod_clean, bins=breakpoints).value_counts().sort_index()
    
    ref_props = ref_counts / len(ref_clean)
    prod_props = prod_counts / len(prod_clean)
    
    ref_props = ref_props.replace(0, 0.0001)
    prod_props = prod_props.replace(0, 0.0001)
    
    psi = np.sum((prod_props - ref_props) * np.log(prod_props / ref_props))
    
    return psi


def jensen_shannon_divergence(ref_data, prod_data, n_bins=10):
    """Calcula la divergencia de Jensen-Shannon"""
    ref_clean = ref_data.dropna()
    prod_clean = prod_data.dropna()
    
    all_data = np.concatenate([ref_clean, prod_clean])
    bins = np.linspace(all_data.min(), all_data.max(), n_bins + 1)
    
    ref_hist, _ = np.histogram(ref_clean, bins=bins)
    prod_hist, _ = np.histogram(prod_clean, bins=bins)
    
    ref_hist = ref_hist / ref_hist.sum()
    prod_hist = prod_hist / prod_hist.sum()
    
    ref_hist = np.where(ref_hist == 0, 1e-10, ref_hist)
    prod_hist = np.where(prod_hist == 0, 1e-10, prod_hist)
    
    js_div = jensenshannon(ref_hist, prod_hist)
    
    return js_div

print("✅ Funciones de detección de drift definidas")

## 📊 3. Análisis de Drift por Variable

In [ ]:
# Identificar tipos de variables
columnas_numericas = df_referencia.select_dtypes(include=[np.number]).columns.tolist()

# Excluir target si existe
if target_col and target_col in columnas_numericas:
    columnas_numericas.remove(target_col)

print(f"📊 Variables numéricas a monitorear: {len(columnas_numericas)}")

In [ ]:
# Analizar drift en variables numéricas
resultados_numericas = []

print("\n" + "="*70)
print("🔍 ANÁLISIS DE DRIFT - VARIABLES NUMÉRICAS")
print("="*70 + "\n")

for col in columnas_numericas:
    print(f"Analizando: {col}")
    
    ref_data = df_referencia[col]
    prod_data = df_produccion[col]
    
    # KS Test
    ks_result = kolmogorov_smirnov_test(ref_data, prod_data)
    
    # PSI
    psi_value = population_stability_index(ref_data, prod_data)
    
    # Jensen-Shannon
    js_value = jensen_shannon_divergence(ref_data, prod_data)
    
    # Determinar drift
    drift_detected = (
        ks_result['drift_detected'] or 
        psi_value > 0.25 or 
        js_value > 0.3
    )
    
    resultado = {
        'Variable': col,
        'KS p-value': ks_result['p_value'],
        'PSI': psi_value,
        'JS Divergence': js_value,
        'Drift Detected': drift_detected
    }
    
    resultados_numericas.append(resultado)
    
    status = "🔴 DRIFT" if drift_detected else "🟢 OK"
    print(f"  {status} - PSI: {psi_value:.4f}, JS: {js_value:.4f}\n")

df_resultados = pd.DataFrame(resultados_numericas)
display(df_resultados.round(4))

## 📋 4. Reporte Final

In [ ]:
print("\n" + "="*70)
print("📋 REPORTE DE MONITOREO")
print("="*70 + "\n")

total_drift = df_resultados['Drift Detected'].sum()
total_vars = len(df_resultados)

print(f"Variables con drift: {total_drift} / {total_vars}")

if total_drift == 0:
    print("\n🟢 ESTADO: SALUDABLE")
    print("   No se detectó drift significativo")
elif total_drift <= total_vars * 0.2:
    print("\n🟡 ESTADO: ATENCIÓN")
    print("   Drift detectado en algunas variables")
else:
    print("\n🔴 ESTADO: CRÍTICO")
    print("   ⚠️ RE-ENTRENAMIENTO RECOMENDADO")

print("\n✅ MONITOREO COMPLETADO")

# 📡 Monitoreo de Modelo y Detección de Data Drift

**Objetivo:** Implementar sistema de monitoreo para detectar degradación del modelo en producción.

Este notebook incluye:
- Detección de Data Drift usando múltiples métricas
- Kolmogorov-Smirnov Test (KS Test)
- Population Stability Index (PSI)
- Jensen-Shannon Divergence
- Análisis de distribuciones de características
- Reportes y recomendaciones

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import joblib
from datetime import datetime

# Estadística
from scipy import stats
from scipy.spatial.distance import jensenshannon
from scipy.stats import ks_2samp, chi2_contingency

warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 📥 1. Cargar Datos de Referencia y Producción

In [ ]:
# Cargar dataset completo (simularemos datos de referencia vs producción)
ruta_datos = os.path.join('..', '..', 'base_de_datos.csv')
df_completo = pd.read_csv(ruta_datos)

print(f"✅ Dataset cargado: {df_completo.shape}")

# Detectar variable objetivo
posibles_targets = [col for col in df_completo.columns if any(keyword in col.lower() for keyword in 
                    ['target', 'label', 'exited', 'churn', 'class'])]
target_col = posibles_targets[0] if posibles_targets else None

if target_col:
    print(f"Variable objetivo: {target_col}")
else:
    print("⚠️ No se detectó variable objetivo")

In [ ]:
# Simular escenario: Dividir en datos de referencia (80%) y datos de producción (20%)
from sklearn.model_selection import train_test_split

df_referencia, df_produccion = train_test_split(
    df_completo, 
    test_size=0.2, 
    random_state=42,
    stratify=df_completo[target_col] if target_col else None
)

print(f"\n📊 División de datos:")
print(f"  - Referencia (baseline): {df_referencia.shape[0]} filas")
print(f"  - Producción (nuevo): {df_produccion.shape[0]} filas")

## 🔍 2. Funciones de Detección de Drift

In [ ]:
def kolmogorov_smirnov_test(ref_data, prod_data, threshold=0.05):
    """Prueba de Kolmogorov-Smirnov para detectar drift"""
    ref_clean = ref_data.dropna()
    prod_clean = prod_data.dropna()
    
    statistic, p_value = ks_2samp(ref_clean, prod_clean)
    
    return {
        'statistic': statistic,
        'p_value': p_value,
        'drift_detected': p_value < threshold,
        'threshold': threshold
    }


def population_stability_index(ref_data, prod_data, n_bins=10):
    """Calcula el Population Stability Index (PSI)"""
    ref_clean = ref_data.dropna()
    prod_clean = prod_data.dropna()
    
    breakpoints = np.linspace(ref_clean.min(), ref_clean.max(), n_bins + 1)
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf
    
    ref_counts = pd.cut(ref_clean, bins=breakpoints).value_counts().sort_index()
    prod_counts = pd.cut(prod_clean, bins=breakpoints).value_counts().sort_index()
    
    ref_props = ref_counts / len(ref_clean)
    prod_props = prod_counts / len(prod_clean)
    
    ref_props = ref_props.replace(0, 0.0001)
    prod_props = prod_props.replace(0, 0.0001)
    
    psi = np.sum((prod_props - ref_props) * np.log(prod_props / ref_props))
    
    return psi


def jensen_shannon_divergence(ref_data, prod_data, n_bins=10):
    """Calcula la divergencia de Jensen-Shannon"""
    ref_clean = ref_data.dropna()
    prod_clean = prod_data.dropna()
    
    all_data = np.concatenate([ref_clean, prod_clean])
    bins = np.linspace(all_data.min(), all_data.max(), n_bins + 1)
    
    ref_hist, _ = np.histogram(ref_clean, bins=bins)
    prod_hist, _ = np.histogram(prod_clean, bins=bins)
    
    ref_hist = ref_hist / ref_hist.sum()
    prod_hist = prod_hist / prod_hist.sum()
    
    ref_hist = np.where(ref_hist == 0, 1e-10, ref_hist)
    prod_hist = np.where(prod_hist == 0, 1e-10, prod_hist)
    
    js_div = jensenshannon(ref_hist, prod_hist)
    
    return js_div

print("✅ Funciones de detección de drift definidas")

## 📊 3. Análisis de Drift por Variable

In [ ]:
# Identificar tipos de variables
columnas_numericas = df_referencia.select_dtypes(include=[np.number]).columns.tolist()

# Excluir target si existe
if target_col and target_col in columnas_numericas:
    columnas_numericas.remove(target_col)

print(f"📊 Variables numéricas a monitorear: {len(columnas_numericas)}")

In [ ]:
# Analizar drift en variables numéricas
resultados_numericas = []

print("\n" + "="*70)
print("🔍 ANÁLISIS DE DRIFT - VARIABLES NUMÉRICAS")
print("="*70 + "\n")

for col in columnas_numericas:
    print(f"Analizando: {col}")
    
    ref_data = df_referencia[col]
    prod_data = df_produccion[col]
    
    # KS Test
    ks_result = kolmogorov_smirnov_test(ref_data, prod_data)
    
    # PSI
    psi_value = population_stability_index(ref_data, prod_data)
    
    # Jensen-Shannon
    js_value = jensen_shannon_divergence(ref_data, prod_data)
    
    # Determinar drift
    drift_detected = (
        ks_result['drift_detected'] or 
        psi_value > 0.25 or 
        js_value > 0.3
    )
    
    resultado = {
        'Variable': col,
        'KS p-value': ks_result['p_value'],
        'PSI': psi_value,
        'JS Divergence': js_value,
        'Drift Detected': drift_detected
    }
    
    resultados_numericas.append(resultado)
    
    status = "🔴 DRIFT" if drift_detected else "🟢 OK"
    print(f"  {status} - PSI: {psi_value:.4f}, JS: {js_value:.4f}\n")

df_resultados = pd.DataFrame(resultados_numericas)
display(df_resultados.round(4))

## 📋 4. Reporte Final

In [ ]:
print("\n" + "="*70)
print("📋 REPORTE DE MONITOREO")
print("="*70 + "\n")

total_drift = df_resultados['Drift Detected'].sum()
total_vars = len(df_resultados)

print(f"Variables con drift: {total_drift} / {total_vars}")

if total_drift == 0:
    print("\n🟢 ESTADO: SALUDABLE")
    print("   No se detectó drift significativo")
elif total_drift <= total_vars * 0.2:
    print("\n🟡 ESTADO: ATENCIÓN")
    print("   Drift detectado en algunas variables")
else:
    print("\n🔴 ESTADO: CRÍTICO")
    print("   ⚠️ RE-ENTRENAMIENTO RECOMENDADO")

print("\n✅ MONITOREO COMPLETADO")

# 📡 Monitoreo de Modelo y Detección de Data Drift

**Objetivo:** Implementar sistema de monitoreo para detectar degradación del modelo en producción.

Este notebook incluye:
- Detección de Data Drift usando múltiples métricas
- Kolmogorov-Smirnov Test (KS Test)
- Population Stability Index (PSI)
- Jensen-Shannon Divergence
- Análisis de distribuciones de características
- Reportes y recomendaciones

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import joblib
from datetime import datetime

# Estadística
from scipy import stats
from scipy.spatial.distance import jensenshannon
from scipy.stats import ks_2samp, chi2_contingency

warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 📥 1. Cargar Datos de Referencia y Producción

In [ ]:
# Cargar dataset completo (simularemos datos de referencia vs producción)
ruta_datos = os.path.join('..', '..', 'base_de_datos.csv')
df_completo = pd.read_csv(ruta_datos)

print(f"✅ Dataset cargado: {df_completo.shape}")

# Detectar variable objetivo
posibles_targets = [col for col in df_completo.columns if any(keyword in col.lower() for keyword in 
                    ['target', 'label', 'exited', 'churn', 'class'])]
target_col = posibles_targets[0] if posibles_targets else None

if target_col:
    print(f"Variable objetivo: {target_col}")
else:
    print("⚠️ No se detectó variable objetivo")

In [ ]:
# Simular escenario: Dividir en datos de referencia (80%) y datos de producción (20%)
# En un caso real, los datos de producción serían datos nuevos recolectados después del despliegue

from sklearn.model_selection import train_test_split

df_referencia, df_produccion = train_test_split(
    df_completo, 
    test_size=0.2, 
    random_state=42,
    stratify=df_completo[target_col] if target_col else None
)

print(f"\n📊 División de datos:")
print(f"  - Referencia (baseline): {df_referencia.shape[0]} filas")
print(f"  - Producción (nuevo): {df_produccion.shape[0]} filas")

# Nota para el usuario
print("\n💡 NOTA: En producción real, 'df_referencia' serían los datos de entrenamiento")
print("   y 'df_produccion' serían los nuevos datos recolectados en producción.")

## 🔍 2. Funciones de Detección de Drift

In [ ]:
def kolmogorov_smirnov_test(ref_data, prod_data, threshold=0.05):
    """
    Prueba de Kolmogorov-Smirnov para detectar drift en variables numéricas.
    
    Args:
        ref_data: Serie de datos de referencia
        prod_data: Serie de datos de producción
        threshold: Nivel de significancia (default 0.05)
    
    Returns:
        dict: Resultado con estadístico, p-value y detección de drift
    """
    # Remover NaN
    ref_clean = ref_data.dropna()
    prod_clean = prod_data.dropna()
    
    # Ejecutar test
    statistic, p_value = ks_2samp(ref_clean, prod_clean)
    
    return {
        'statistic': statistic,
        'p_value': p_value,
        'drift_detected': p_value < threshold,
        'threshold': threshold
    }


def population_stability_index(ref_data, prod_data, n_bins=10):
    """
    Calcula el Population Stability Index (PSI) para detectar drift.
    
    PSI < 0.1: Sin cambio significativo
    PSI 0.1-0.25: Cambio moderado
    PSI > 0.25: Cambio significativo (drift detectado)
    
    Args:
        ref_data: Serie de datos de referencia
        prod_data: Serie de datos de producción
        n_bins: Número de bins para discretización
    
    Returns:
        float: Valor PSI
    """
    # Remover NaN
    ref_clean = ref_data.dropna()
    prod_clean = prod_data.dropna()
    
    # Crear bins basados en los datos de referencia
    breakpoints = np.linspace(ref_clean.min(), ref_clean.max(), n_bins + 1)
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf
    
    # Calcular proporciones en cada bin
    ref_counts = pd.cut(ref_clean, bins=breakpoints).value_counts().sort_index()
    prod_counts = pd.cut(prod_clean, bins=breakpoints).value_counts().sort_index()
    
    ref_props = ref_counts / len(ref_clean)
    prod_props = prod_counts / len(prod_clean)
    
    # Evitar divisiones por cero
    ref_props = ref_props.replace(0, 0.0001)
    prod_props = prod_props.replace(0, 0.0001)
    
    # Calcular PSI
    psi = np.sum((prod_props - ref_props) * np.log(prod_props / ref_props))
    
    return psi


def jensen_shannon_divergence(ref_data, prod_data, n_bins=10):
    """
    Calcula la divergencia de Jensen-Shannon entre dos distribuciones.
    
    JS < 0.1: Distribuciones similares
    JS 0.1-0.3: Diferencia moderada
    JS > 0.3: Diferencia significativa (drift)
    
    Args:
        ref_data: Serie de datos de referencia
        prod_data: Serie de datos de producción
        n_bins: Número de bins para discretización
    
    Returns:
        float: Divergencia JS
    """
    # Remover NaN
    ref_clean = ref_data.dropna()
    prod_clean = prod_data.dropna()
    
    # Crear bins
    all_data = np.concatenate([ref_clean, prod_clean])
    bins = np.linspace(all_data.min(), all_data.max(), n_bins + 1)
    
    # Calcular histogramas normalizados
    ref_hist, _ = np.histogram(ref_clean, bins=bins)
    prod_hist, _ = np.histogram(prod_clean, bins=bins)
    
    # Normalizar
    ref_hist = ref_hist / ref_hist.sum()
    prod_hist = prod_hist / prod_hist.sum()
    
    # Evitar ceros
    ref_hist = np.where(ref_hist == 0, 1e-10, ref_hist)
    prod_hist = np.where(prod_hist == 0, 1e-10, prod_hist)
    
    # Calcular divergencia
    js_div = jensenshannon(ref_hist, prod_hist)
    
    return js_div


def chi_square_test_categorical(ref_data, prod_data, threshold=0.05):
    """
    Test Chi-cuadrado para variables categóricas.
    
    Args:
        ref_data: Serie de datos de referencia
        prod_data: Serie de datos de producción
        threshold: Nivel de significancia
    
    Returns:
        dict: Resultado del test
    """
    # Crear tabla de contingencia
    ref_counts = ref_data.value_counts()
    prod_counts = prod_data.value_counts()
    
    # Combinar todas las categorías
    all_categories = set(ref_counts.index) | set(prod_counts.index)
    
    ref_freq = [ref_counts.get(cat, 0) for cat in all_categories]
    prod_freq = [prod_counts.get(cat, 0) for cat in all_categories]
    
    # Test chi-cuadrado
    contingency_table = np.array([ref_freq, prod_freq])
    
    try:
        chi2, p_value, dof, expected = chi2_contingency(contingency_table)
        return {
            'chi2': chi2,
            'p_value': p_value,
            'drift_detected': p_value < threshold,
            'threshold': threshold
        }
    except:
        return {
            'chi2': None,
            'p_value': None,
            'drift_detected': False,
            'threshold': threshold
        }

print("✅ Funciones de detección de drift definidas")

## 📊 3. Análisis de Drift por Variable

In [ ]:
# Identificar tipos de variables
columnas_numericas = df_referencia.select_dtypes(include=[np.number]).columns.tolist()
columnas_categoricas = df_referencia.select_dtypes(include=['object', 'category']).columns.tolist()

# Excluir target si existe
if target_col:
    if target_col in columnas_numericas:
        columnas_numericas.remove(target_col)
    if target_col in columnas_categoricas:
        columnas_categoricas.remove(target_col)

print(f"📊 Variables a monitorear:")
print(f"  - Numéricas: {len(columnas_numericas)}")
print(f"  - Categóricas: {len(columnas_categoricas)}")

In [ ]:
# Analizar drift en variables numéricas
resultados_numericas = []

print("\n" + "="*70)
print("🔍 ANÁLISIS DE DRIFT - VARIABLES NUMÉRICAS")
print("="*70 + "\n")

for col in columnas_numericas:
    print(f"\nAnalizando: {col}")
    
    ref_data = df_referencia[col]
    prod_data = df_produccion[col]
    
    # KS Test
    ks_result = kolmogorov_smirnov_test(ref_data, prod_data)
    
    # PSI
    psi_value = population_stability_index(ref_data, prod_data)
    
    # Jensen-Shannon
    js_value = jensen_shannon_divergence(ref_data, prod_data)
    
    # Determinar drift general
    drift_detected = (
        ks_result['drift_detected'] or 
        psi_value > 0.25 or 
        js_value > 0.3
    )
    
    resultado = {
        'Variable': col,
        'KS Statistic': ks_result['statistic'],
        'KS p-value': ks_result['p_value'],
        'PSI': psi_value,
        'JS Divergence': js_value,
        'Drift Detected': drift_detected
    }
    
    resultados_numericas.append(resultado)
    
    # Mostrar resultado
    status = "🔴 DRIFT DETECTADO" if drift_detected else "🟢 Sin drift"
    print(f"  {status}")
    print(f"  - KS p-value: {ks_result['p_value']:.4f}")
    print(f"  - PSI: {psi_value:.4f}")
    print(f"  - JS Divergence: {js_value:.4f}")

# Crear DataFrame de resultados
df_resultados_num = pd.DataFrame(resultados_numericas)

print("\n" + "="*70)
print("📊 RESUMEN - VARIABLES NUMÉRICAS")
print("="*70 + "\n")
display(df_resultados_num.round(4))

In [ ]:
# Analizar drift en variables categóricas
resultados_categoricas = []

print("\n" + "="*70)
print("🔍 ANÁLISIS DE DRIFT - VARIABLES CATEGÓRICAS")
print("="*70 + "\n")

for col in columnas_categoricas:
    print(f"\nAnalizando: {col}")
    
    ref_data = df_referencia[col]
    prod_data = df_produccion[col]
    
    # Chi-square Test
    chi2_result = chi_square_test_categorical(ref_data, prod_data)
    
    drift_detected = chi2_result['drift_detected']
    
    resultado = {
        'Variable': col,
        'Chi2 Statistic': chi2_result['chi2'],
        'Chi2 p-value': chi2_result['p_value'],
        'Drift Detected': drift_detected
    }
    
    resultados_categoricas.append(resultado)
    
    # Mostrar resultado
    status = "🔴 DRIFT DETECTADO" if drift_detected else "🟢 Sin drift"
    print(f"  {status}")
    if chi2_result['p_value'] is not None:
        print(f"  - Chi2 p-value: {chi2_result['p_value']:.4f}")

# Crear DataFrame de resultados
if resultados_categoricas:
    df_resultados_cat = pd.DataFrame(resultados_categoricas)
    
    print("\n" + "="*70)
    print("📊 RESUMEN - VARIABLES CATEGÓRICAS")
    print("="*70 + "\n")
    display(df_resultados_cat.round(4))
else:
    print("\n⚠️ No hay variables categóricas para analizar")
    df_resultados_cat = pd.DataFrame()

## 📈 4. Visualización de Drift

In [ ]:
# Visualizar distribuciones de variables con drift detectado
variables_con_drift = df_resultados_num[df_resultados_num['Drift Detected'] == True]['Variable'].tolist()

if variables_con_drift:
    print(f"\n🔴 Se detectó drift en {len(variables_con_drift)} variables numéricas:\n")
    for var in variables_con_drift:
        print(f"  - {var}")
    
    # Visualizar cada variable con drift
    for col in variables_con_drift[:5]:  # Máximo 5 para no sobrecargar
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Histograma comparativo
        axes[0].hist(df_referencia[col].dropna(), bins=30, alpha=0.6, label='Referencia', edgecolor='black')
        axes[0].hist(df_produccion[col].dropna(), bins=30, alpha=0.6, label='Producción', edgecolor='black')
        axes[0].set_xlabel(col)
        axes[0].set_ylabel('Frecuencia')
        axes[0].set_title(f'Distribución Comparativa: {col}')
        axes[0].legend()
        axes[0].grid(alpha=0.3)
        
        # Boxplot comparativo
        data_plot = pd.DataFrame({
            'Valor': np.concatenate([df_referencia[col].dropna(), df_produccion[col].dropna()]),
            'Conjunto': ['Referencia'] * len(df_referencia[col].dropna()) + 
                       ['Producción'] * len(df_produccion[col].dropna())
        })
        
        sns.boxplot(data=data_plot, x='Conjunto', y='Valor', ax=axes[1])
        axes[1].set_title(f'Boxplot Comparativo: {col}')
        axes[1].grid(alpha=0.3)
        
        plt.tight_layout()
        plt.show()
else:
    print("\n🟢 No se detectó drift significativo en variables numéricas")

In [ ]:
# Mapa de calor de métricas de drift
if len(df_resultados_num) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # PSI
    df_plot = df_resultados_num.sort_values('PSI', ascending=False).head(10)
    axes[0].barh(df_plot['Variable'], df_plot['PSI'], edgecolor='black', alpha=0.7)
    axes[0].axvline(x=0.1, color='orange', linestyle='--', label='Moderado (0.1)')
    axes[0].axvline(x=0.25, color='red', linestyle='--', label='Significativo (0.25)')
    axes[0].set_xlabel('PSI')
    axes[0].set_title('Population Stability Index (Top 10)')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # JS Divergence
    df_plot = df_resultados_num.sort_values('JS Divergence', ascending=False).head(10)
    axes[1].barh(df_plot['Variable'], df_plot['JS Divergence'], edgecolor='black', alpha=0.7, color='green')
    axes[1].axvline(x=0.1, color='orange', linestyle='--', label='Moderado (0.1)')
    axes[1].axvline(x=0.3, color='red', linestyle='--', label='Significativo (0.3)')
    axes[1].set_xlabel('JS Divergence')
    axes[1].set_title('Jensen-Shannon Divergence (Top 10)')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    # KS p-value
    df_plot = df_resultados_num.sort_values('KS p-value', ascending=True).head(10)
    axes[2].barh(df_plot['Variable'], df_plot['KS p-value'], edgecolor='black', alpha=0.7, color='purple')
    axes[2].axvline(x=0.05, color='red', linestyle='--', label='Umbral (0.05)')
    axes[2].set_xlabel('KS p-value')
    axes[2].set_title('Kolmogorov-Smirnov p-value (Top 10)')
    axes[2].legend()
    axes[2].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 📊 5. Análisis de la Variable Objetivo

In [ ]:
if target_col:
    print("\n" + "="*70)
    print(f"🎯 ANÁLISIS DE DRIFT - VARIABLE OBJETIVO ({target_col})")
    print("="*70 + "\n")
    
    # Distribución de la variable objetivo
    ref_target = df_referencia[target_col]
    prod_target = df_produccion[target_col]
    
    print("📊 Distribución - Referencia:")
    display(ref_target.value_counts(normalize=True).round(4))
    
    print("\n📊 Distribución - Producción:")
    display(prod_target.value_counts(normalize=True).round(4))
    
    # Test estadístico
    if pd.api.types.is_numeric_dtype(ref_target):
        # Variable numérica - usar KS test
        ks_result = kolmogorov_smirnov_test(ref_target, prod_target)
        print(f"\nKS Test:")
        print(f"  - Statistic: {ks_result['statistic']:.4f}")
        print(f"  - p-value: {ks_result['p_value']:.4f}")
        drift = ks_result['drift_detected']
    else:
        # Variable categórica - usar Chi-square
        chi2_result = chi_square_test_categorical(ref_target, prod_target)
        print(f"\nChi-Square Test:")
        if chi2_result['p_value'] is not None:
            print(f"  - Chi2: {chi2_result['chi2']:.4f}")
            print(f"  - p-value: {chi2_result['p_value']:.4f}")
        drift = chi2_result['drift_detected']
    
    # Resultado
    if drift:
        print("\n🔴 ALERTA: Se detectó drift en la variable objetivo")
        print("   Esto puede indicar cambios en el problema de negocio o en la población objetivo.")
    else:
        print("\n🟢 No se detectó drift significativo en la variable objetivo")
    
    # Visualización
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Gráfico de barras
    ref_counts = ref_target.value_counts(normalize=True)
    prod_counts = prod_target.value_counts(normalize=True)
    
    x = np.arange(len(ref_counts))
    width = 0.35
    
    axes[0].bar(x - width/2, ref_counts.values, width, label='Referencia', alpha=0.7)
    axes[0].bar(x + width/2, prod_counts.values, width, label='Producción', alpha=0.7)
    axes[0].set_xlabel(target_col)
    axes[0].set_ylabel('Proporción')
    axes[0].set_title(f'Distribución de {target_col}')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(ref_counts.index)
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Diferencias absolutas
    diffs = (prod_counts - ref_counts).abs()
    axes[1].bar(range(len(diffs)), diffs.values, edgecolor='black', alpha=0.7, color='red')
    axes[1].set_xlabel('Clase')
    axes[1].set_ylabel('Diferencia Absoluta')
    axes[1].set_title('Diferencia en Proporciones')
    axes[1].set_xticks(range(len(diffs)))
    axes[1].set_xticklabels(diffs.index)
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("\n⚠️ No se especificó variable objetivo")

## 📋 6. Reporte Final y Recomendaciones

In [ ]:
print("\n" + "="*70)
print("📋 REPORTE FINAL DE MONITOREO")
print("="*70)

# Resumen general
total_variables = len(df_resultados_num) + len(df_resultados_cat)
drift_numericas = df_resultados_num['Drift Detected'].sum()
drift_categoricas = df_resultados_cat['Drift Detected'].sum() if len(df_resultados_cat) > 0 else 0
total_drift = drift_numericas + drift_categoricas

print(f"\n📊 RESUMEN EJECUTIVO:")
print(f"  - Total de variables monitoreadas: {total_variables}")
print(f"  - Variables con drift detectado: {total_drift} ({total_drift/total_variables*100:.1f}%)")
print(f"    • Numéricas: {drift_numericas} / {len(df_resultados_num)}")
print(f"    • Categóricas: {drift_categoricas} / {len(df_resultados_cat) if len(df_resultados_cat) > 0 else 0}")

print(f"\n📅 Fecha del análisis: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📦 Tamaño de referencia: {len(df_referencia)} muestras")
print(f"📦 Tamaño de producción: {len(df_produccion)} muestras")

# Recomendaciones
print("\n" + "="*70)
print("💡 RECOMENDACIONES")
print("="*70 + "\n")

if total_drift == 0:
    print("🟢 ESTADO: SALUDABLE")
    print("\n  No se detectó drift significativo en ninguna variable.")
    print("  El modelo está operando dentro de los parámetros esperados.")
    print("\n  Acciones sugeridas:")
    print("    • Continuar con el monitoreo periódico")
    print("    • Mantener el modelo en producción sin cambios")

elif total_drift <= total_variables * 0.2:
    print("🟡 ESTADO: ATENCIÓN")
    print("\n  Se detectó drift en algunas variables, pero es manejable.")
    print("\n  Acciones sugeridas:")
    print("    • Incrementar la frecuencia de monitoreo")
    print("    • Analizar las variables afectadas en detalle")
    print("    • Considerar re-entrenamiento si el desempeño se degrada")
    print("    • Documentar los cambios observados")

else:
    print("🔴 ESTADO: CRÍTICO")
    print("\n  Se detectó drift significativo en múltiples variables.")
    print("  ⚠️ El modelo puede no estar funcionando correctamente.")
    print("\n  Acciones requeridas:")
    print("    • ⚡ RE-ENTRENAR EL MODELO con datos actualizados")
    print("    • Investigar las causas del drift (cambios de negocio, población, etc.)")
    print("    • Evaluar si el modelo actual sigue siendo apropiado")
    print("    • Considerar implementar un nuevo pipeline de features")
    print("    • Notificar a stakeholders sobre el estado del modelo")

# Variables específicas con drift
if total_drift > 0:
    print("\n📌 VARIABLES QUE REQUIEREN ATENCIÓN:\n")
    
    if drift_numericas > 0:
        vars_drift_num = df_resultados_num[df_resultados_num['Drift Detected'] == True]
        print("  Variables numéricas:")
        for idx, row in vars_drift_num.iterrows():
            print(f"    • {row['Variable']}:")
            print(f"        PSI = {row['PSI']:.4f}, JS = {row['JS Divergence']:.4f}")
    
    if drift_categoricas > 0:
        vars_drift_cat = df_resultados_cat[df_resultados_cat['Drift Detected'] == True]
        print("\n  Variables categóricas:")
        for idx, row in vars_drift_cat.iterrows():
            print(f"    • {row['Variable']}")

print("\n" + "="*70)
print("✅ MONITOREO COMPLETADO")
print("="*70)

## 💾 7. Guardar Reporte

In [ ]:
# Guardar resultados en archivo
ruta_reporte = os.path.join('..', '..', f'drift_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv')

# Combinar resultados
df_reporte_completo = pd.concat([
    df_resultados_num[['Variable', 'PSI', 'JS Divergence', 'KS p-value', 'Drift Detected']],
    df_resultados_cat[['Variable', 'Chi2 p-value', 'Drift Detected']] if len(df_resultados_cat) > 0 else pd.DataFrame()
])

df_reporte_completo.to_csv(ruta_reporte, index=False)

print(f"\n💾 Reporte guardado en: {ruta_reporte}")
print("\n📊 Resumen del reporte:")
display(df_reporte_completo.head(10))

print("\n" + "="*70)
print("✅ PIPELINE DE MONITOREO COMPLETADO")
print("="*70)
print("\n📌 Este notebook debe ejecutarse periódicamente para:")
print("  • Detectar degradación del modelo")
print("  • Identificar cambios en los datos de entrada")
print("  • Tomar decisiones de re-entrenamiento")
print("  • Mantener la calidad del modelo en producción")